# 🗄️ RAG 3: SQL Database RAG with Natural Language Queries
## Project: Build RAG using AWS Bedrock & SageMaker Notebook

### What is this notebook?
This notebook builds a **Text-to-SQL RAG system** — it allows business users to query a relational database using plain English, without writing any SQL. It uses:
- **SQLDatabase (LangChain)** — connects to SQLite and introspects the schema
- **SQLDatabaseChain** — the chain that translates natural language to SQL and executes it
- **Amazon Nova Lite via Bedrock** — the LLM that writes the SQL queries
- **Amazon Titan Embeddings** — for semantic similarity testing
- **LangSmith** — to trace every SQL generation and execution step

### Why SQL RAG?
Most business data lives in relational databases. Non-technical users need analysts or engineers to extract data. SQL RAG eliminates this bottleneck — anyone can ask questions like *"Which employee has the most customers?"* and get instant answers.

### The Chinook Database
We use the **Chinook** database — a sample music store database with 11 tables including Artists, Albums, Tracks, Customers, Invoices, and Employees. It is a standard benchmark for SQL tasks.


## Cell 1: Install Required Packages
### Why this cell?
SQL RAG requires a few additional packages beyond the basic RAG setup:
- **sqlalchemy** — Python SQL toolkit; provides the database connection abstraction
- **langchain-experimental** — contains `SQLDatabaseChain` (not in main langchain package)
- **langchain-community** — provides `SQLDatabase` utility class

### What we do here:
Install all dependencies in one command to avoid partial installation issues.


In [1]:
# =============================================================================
# CELL 1: INSTALL REQUIRED PACKAGES
# =============================================================================
print("📦 Installing required packages for SQL RAG system...")

import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '--upgrade',
    'langsmith', 'langchain-community', 'langchain-aws', 'sqlalchemy',
    'langchain-core', 'langchain', 'langchain-experimental',
    '--quiet'
], check=True)

print("✅ All required packages installed successfully!")

📦 Installing required packages for SQL RAG system...
✅ All required packages installed successfully!


## Cell 2: Configure LangSmith Tracing
### Why this cell?
For the SQL RAG system, LangSmith tracing is especially valuable because it captures:
- The **natural language question** received
- The **SQL query generated** by the LLM
- The **SQL result** returned from the database
- The **final natural language answer**

This full trace makes it easy to debug incorrect SQL or unexpected results.

### What we do here:
Set the LangSmith project to **"K21"** — the same project used in notebooks 1 and 2. All three notebooks will send traces to the same project for a unified view.


In [7]:
import os
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"]    = "K21"
os.environ["LANGCHAIN_API_KEY"]    = "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"

print("✅ LangSmith ready — Project: K21")
print("🔗 https://smith.langchain.com → Tracing → K21")

✅ LangSmith ready — Project: K21
🔗 https://smith.langchain.com → Tracing → K21


## Cell 3: Configure AWS Bedrock Client
### Why this cell?
The Bedrock client provides access to the Nova Lite LLM that will generate SQL queries. The same client will also be used for Amazon Titan embeddings later in this notebook.

### IAM Permissions Required:
The SageMaker notebook IAM role needs `AmazonBedrockFullAccess` attached. If you get an `AccessDeniedException`, check the IAM role in the SageMaker console.

### What we do here:
Initialize the boto3 Bedrock runtime client for the `us-east-1` region.


In [8]:
# =============================================================================
# CELL 3: CONFIGURE AWS BEDROCK CLIENT
# =============================================================================
import boto3

print("⚙️ Setting up AWS Bedrock client...")
AWS_REGION = "us-east-1"

bedrock_client = boto3.client(
    service_name="bedrock-runtime",
    region_name=AWS_REGION,
)

print(f"✅ AWS Bedrock client ready! Region: {AWS_REGION}")

⚙️ Setting up AWS Bedrock client...
✅ AWS Bedrock client ready! Region: us-east-1


## Cell 4: Connect to SQL Database
### Why this cell?
`SQLDatabase` is LangChain's database abstraction layer. When initialized, it:
1. Connects to the database via SQLAlchemy
2. **Automatically reads the schema** — table names, column names, data types, sample rows
3. Formats this schema information to be injected into the LLM prompt

This schema injection is what enables the LLM to write correct SQL — it "knows" the database structure before generating any query.

### The Chinook Database Tables:
`Album`, `Artist`, `Customer`, `Employee`, `Genre`, `Invoice`, `InvoiceLine`, `MediaType`, `Playlist`, `PlaylistTrack`, `Track`

### What we do here:
Connect to `Chinook.db`, list all available tables, and run a sample raw SQL query to confirm connectivity.


In [9]:
# =============================================================================
# CELL 4: CONNECT TO SQL DATABASE (Chinook)
# =============================================================================
from langchain_community.utilities import SQLDatabase

print("🗄️ Connecting to SQLite Chinook database...")
db = SQLDatabase.from_uri("sqlite:///Chinook.db")

print(f"✅ Connected! Dialect: {db.dialect}")
table_names = db.get_usable_table_names()
print(f"📊 Tables available ({len(table_names)}): {table_names}")

# Test raw SQL query
print("\n🧪 Testing raw SQL query:")
sample = db.run("SELECT ArtistId, Name FROM Artist LIMIT 5;")
print(f"   Result: {sample}")
print(f"\n🎯 Database ready!")

🗄️ Connecting to SQLite Chinook database...
✅ Connected! Dialect: sqlite
📊 Tables available (11): ['Album', 'Artist', 'Customer', 'Employee', 'Genre', 'Invoice', 'InvoiceLine', 'MediaType', 'Playlist', 'PlaylistTrack', 'Track']

🧪 Testing raw SQL query:
   Result: [(1, 'AC/DC'), (2, 'Accept'), (3, 'Aerosmith'), (4, 'Alanis Morissette'), (5, 'Alice In Chains')]

🎯 Database ready!


## Cell 5: Initialize Language Model
### Why this cell?
The LLM in this notebook has a specialized task — **writing SQL queries** from natural language. It needs to:
1. Understand the database schema (provided in context)
2. Translate the user question into valid SQL for that schema
3. Interpret the SQL result and provide a natural language answer

### Why Amazon Nova Lite?
`amazon.nova-lite-v1:0` is a strong reasoning model well-suited for text-to-SQL tasks. The original notebook used `anthropic.claude-3-sonnet-20240229-v1:0` which has reached **End of Life** and returns `ResourceNotFoundException`.

### What we do here:
Initialize `ChatBedrockConverse` with Nova Lite and verify it responds correctly with a simple test.


In [10]:
# =============================================================================
# CELL 5: INITIALIZE LANGUAGE MODEL
# FIXED: ChatBedrockConverse + nova-lite replaces EOL ChatBedrock + claude-3-sonnet
# =============================================================================
from langchain_aws import ChatBedrockConverse

print("🤖 Initializing language model...")

llm = ChatBedrockConverse(
    client=bedrock_client,
    model="amazon.nova-lite-v1:0"
)

# Verify LLM is working
test = llm.invoke("Say hello").content
print(f"✅ LLM ready! Test: {test}")
print("🔧 Model: amazon.nova-lite-v1:0 (ChatBedrockConverse)")

🤖 Initializing language model...
✅ LLM ready! Test: Hello! How can I assist you today? If you have any questions or need help with something, feel free to let me know. Whether it's about a topic you're curious about, a problem you're facing, or just want to chat, I'm here to help.
🔧 Model: amazon.nova-lite-v1:0 (ChatBedrockConverse)


## Cell 6: Create SQL Query Chain
### Why this cell?
`SQLDatabaseChain` is the core component that:
1. Receives a natural language question
2. Builds a prompt with the question + database schema
3. Sends prompt to the LLM → receives SQL query
4. Executes the SQL against the database
5. Sends the result back to the LLM for a natural language answer
6. Returns the final answer

### verbose=True:
Setting `verbose=True` shows the intermediate steps — the generated SQL query and the raw result. This is useful for debugging and learning.

### Why langchain-experimental?
`SQLDatabaseChain` was moved to `langchain-experimental` because it executes arbitrary LLM-generated SQL — which could be dangerous in production without proper guardrails. It's appropriate for this lab context.

### What we do here:
Create the SQL chain and test it with a simple business question.


In [11]:
# =============================================================================
# CELL 6: CREATE SQL QUERY CHAIN
# FIXED: Custom prompt stops Nova Lite from wrapping SQL in markdown backticks
# =============================================================================
from langchain_experimental.sql import SQLDatabaseChain
from langchain_core.prompts import PromptTemplate

print("🔗 Creating SQL query generation chain...")

# Custom prompt explicitly forbids markdown formatting in SQL output
_PROMPT = PromptTemplate(
    input_variables=["input", "table_info", "dialect"],
    template="""You are a {dialect} expert. Given an input question, create a
syntactically correct {dialect} query to run, then look at the results and
return the answer to the input question.

CRITICAL RULES:
- Return ONLY the raw SQL query with NO markdown, NO backticks, NO ```sql``` wrapping
- Do NOT write SELECT * — only select relevant columns
- Use LIMIT 10 unless the question asks for a specific number

Table info:
{table_info}

Question: {input}
SQLQuery:"""
)

sql_chain = SQLDatabaseChain.from_llm(
    llm,
    db,
    verbose=True,
    prompt=_PROMPT
)

print("✅ SQL chain created!")
print("🔧 Custom prompt: prevents markdown-wrapped SQL")
print("\n🧪 Testing with a simple question:")

question = "For how many customers is the company value missing?"
print(f"❓ Question: {question}")

try:
    result = sql_chain.invoke({"query": question})
    print(f"\n✅ Answer: {result.get('result', 'No result')}")
except Exception as e:
    print(f"❌ Error: {e}")

🔗 Creating SQL query generation chain...
✅ SQL chain created!
🔧 Custom prompt: prevents markdown-wrapped SQL

🧪 Testing with a simple question:
❓ Question: For how many customers is the company value missing?


> Entering new SQLDatabaseChain chain...
For how many customers is the company value missing?
SQLQuery:SELECT COUNT(*) FROM Customer WHERE Company IS NULL
SQLResult: [(49,)]
Answer:SELECT COUNT(*) FROM Customer WHERE Company IS NULL
> Finished chain.

✅ Answer: SELECT COUNT(*) FROM Customer WHERE Company IS NULL


## Cell 7: Initialize Titan Embeddings
### Why this cell?
Even in a SQL RAG system, embeddings are useful for:
- **Question routing** — determining whether a question is for SQL or documents
- **Semantic caching** — caching results for semantically similar questions
- **Hybrid search** — combining SQL results with document retrieval

### Amazon Titan Embed Text v2:
- Produces 1024-dimensional dense vectors
- Optimized for English semantic similarity
- Directly available via AWS Bedrock — no external API needed
- Significantly higher quality than older 384-dim models

### What we do here:
Initialize the embeddings model and run a quick test to verify it produces the correct vector dimensions.


In [12]:
# =============================================================================
# CELL 7: INITIALIZE TITAN EMBEDDINGS
# =============================================================================
from langchain_aws.embeddings.bedrock import BedrockEmbeddings

print("🔧 Initializing Amazon Titan Embed Text v2...")

embeddings = BedrockEmbeddings(
    client=bedrock_client,
    model_id="amazon.titan-embed-text-v2:0"
)

# Test embeddings
sample_texts = [
    "What are the business applications of Amazon Q?",
    "How does Amazon Q integrate with AWS?"
]

result = embeddings.embed_documents(sample_texts)
print(f"✅ Embeddings ready!")
print(f"📊 Dimensions: {len(result[0])} (Amazon Titan v2)")
print(f"🔢 Sample values: {result[0][:4]}...")

🔧 Initializing Amazon Titan Embed Text v2...
✅ Embeddings ready!
📊 Dimensions: 1024 (Amazon Titan v2)
🔢 Sample values: [-0.0780167430639267, 0.015050669200718403, -0.04666610807180405, -0.038271233439445496]...


## Cell 8: Basic SQL Query Testing
### Why this cell?
Before running complex business queries, we validate the SQL chain with straightforward factual questions. This helps confirm:
- The LLM correctly reads the schema
- Generated SQL is syntactically valid for SQLite
- Results are returned and interpreted correctly

### What we do here:
Run 5 basic questions covering different tables and query patterns (COUNT, SELECT, WHERE, ORDER BY).


In [13]:
# =============================================================================
# CELL 8: BASIC SQL QUERY TESTING
# =============================================================================
print("🧪 Basic SQL RAG Testing")
print("=" * 60)

basic_questions = [
    "How many customers are in the database?",
    "How many tracks are in the database?",
    "Which artist has the most albums?",
    "How many employees are there?",
    "For how many customers is the company value missing?"
]

for i, question in enumerate(basic_questions, 1):
    print(f"\n{i}. ❓ {question}")
    try:
        result = sql_chain.invoke({"query": question})
        print(f"   💡 {result.get('result', 'No result')}")
    except Exception as e:
        print(f"   ❌ Error: {e}")
    print("   " + "-" * 50)

print("\n✅ Basic testing complete!")

🧪 Basic SQL RAG Testing

1. ❓ How many customers are in the database?


> Entering new SQLDatabaseChain chain...
How many customers are in the database?
SQLQuery:SELECT COUNT(*) FROM Customer
SQLResult: [(59,)]
Answer:SELECT COUNT(*) FROM Customer
SQLResult:
> Finished chain.
   💡 SELECT COUNT(*) FROM Customer
SQLResult:
   --------------------------------------------------

2. ❓ How many tracks are in the database?


> Entering new SQLDatabaseChain chain...
How many tracks are in the database?
SQLQuery:SELECT COUNT(TrackId) AS TotalTracks FROM Track;
SQLResult: [(3503,)]
Answer:SELECT COUNT(TrackId) AS TotalTracks FROM Track;
> Finished chain.
   💡 SELECT COUNT(TrackId) AS TotalTracks FROM Track;
   --------------------------------------------------

3. ❓ Which artist has the most albums?


> Entering new SQLDatabaseChain chain...
Which artist has the most albums?
SQLQuery:SELECT Artist.Name, COUNT(Album.AlbumId) AS NumberOfAlbums
FROM Album
JOIN Artist ON Album.ArtistId = Artist.Arti

## Cell 9: Advanced Business Intelligence Queries
### Why this cell?
Real business value comes from complex analytical queries — multi-table JOINs, GROUP BY aggregations, ORDER BY rankings. This cell tests whether the LLM can generate correct complex SQL from business language.

### What these queries test:
| Query | SQL Pattern |
|---|---|
| Total sales by country | JOIN + GROUP BY + SUM |
| Employee with most customers | JOIN + COUNT + ORDER BY |
| Top tracks by sales | JOIN + GROUP BY + ORDER BY |
| Average invoice by year | strftime + GROUP BY + AVG |
| Genre with most tracks | JOIN + COUNT + LIMIT 1 |

### What we do here:
Run 5 advanced analytical questions and display the business insights generated.


In [14]:
# =============================================================================
# CELL 9: ADVANCED BUSINESS INTELLIGENCE QUERIES
# =============================================================================
print("🧪 Advanced Business Intelligence Testing")
print("=" * 70)

advanced_questions = [
    "What is the total sales amount by country? Show top 5.",
    "Which employee has the most customers?",
    "List the top 10 tracks by sales quantity.",
    "What is the average invoice total by year?",
    "Which genre has the most tracks?"
]

for i, question in enumerate(advanced_questions, 1):
    print(f"\n{i}. ❓ {question}")
    try:
        result = sql_chain.invoke({"query": question})
        print(f"   📊 {result.get('result', 'No result')}")
    except Exception as e:
        print(f"   ❌ Error: {e}")
    print("   " + "-" * 60)

print("\n🎉 Advanced testing complete!")

🧪 Advanced Business Intelligence Testing

1. ❓ What is the total sales amount by country? Show top 5.


> Entering new SQLDatabaseChain chain...
What is the total sales amount by country? Show top 5.
SQLQuery:SELECT c.Country, SUM(i.Total) AS TotalSales
FROM Invoice i
JOIN Customer c ON i.CustomerId = c.CustomerId
GROUP BY c.Country
ORDER BY TotalSales DESC
LIMIT 5
SQLResult: [('USA', 523.06), ('Canada', 303.96), ('France', 195.1), ('Brazil', 190.1), ('Germany', 156.48)]
Answer:SELECT c.Country, SUM(i.Total) AS TotalSales
FROM Invoice i
JOIN Customer c ON i.CustomerId = c.CustomerId
GROUP BY c.Country
ORDER BY TotalSales DESC
LIMIT 5
> Finished chain.
   📊 SELECT c.Country, SUM(i.Total) AS TotalSales
FROM Invoice i
JOIN Customer c ON i.CustomerId = c.CustomerId
GROUP BY c.Country
ORDER BY TotalSales DESC
LIMIT 5
   ------------------------------------------------------------

2. ❓ Which employee has the most customers?


> Entering new SQLDatabaseChain chain...
Which employee has the m

## Cell 10: Interactive SQL Mode
### Why this cell?
The interactive mode allows users to type any natural language question and receive an immediate SQL-powered answer. This demonstrates the end-user experience of the SQL RAG system.

### How it works:
1. User types a plain English question
2. `sql_chain.invoke()` generates SQL, executes it, and formats the answer
3. The loop continues until the user types 'quit'

### What we do here:
Define the interactive function. Uncomment the last line to start interactive querying.


In [15]:
# =============================================================================
# CELL 10: INTERACTIVE SQL QUERY MODE
# =============================================================================
def interactive_sql_mode():
    print("\n" + "=" * 60)
    print("🤖 SQL RAG — INTERACTIVE MODE")
    print("=" * 60)
    print("Ask any question about the Chinook music database.")
    print("Type 'quit' to exit.\n")

    count = 0
    while True:
        user_input = input(f"[{count+1}] ❓ Your question: ").strip()
        if user_input.lower() in ['quit', 'exit', 'q']:
            print(f"\n👋 Goodbye! Answered {count} questions.")
            break
        if not user_input:
            print("⚠️  Please enter a question.")
            continue
        try:
            result = sql_chain.invoke({"query": user_input})
            print(f"📊 {result.get('result', 'No result')}\n")
            count += 1
        except Exception as e:
            print(f"❌ Error: {e}\n")

print("✅ Interactive mode defined!")
print("💡 Uncomment the line below to start:")
print("# interactive_sql_mode()")
# interactive_sql_mode()

✅ Interactive mode defined!
💡 Uncomment the line below to start:
# interactive_sql_mode()


## Cell 11: System Summary & Final Verification
### Why this cell?
The summary cell provides:
1. A complete health check of all system components
2. A final test query to confirm end-to-end operation
3. A link to the LangSmith dashboard for trace review

### What we do here:
Print the status of every component and run a definitive final query to confirm the system is fully operational.


In [16]:
# =============================================================================
# CELL 11: SYSTEM SUMMARY & FINAL VERIFICATION
# =============================================================================
print("📋 SQL RAG SYSTEM — FINAL SUMMARY")
print("=" * 60)

print("\n🔧 COMPONENT STATUS:")
print(f"   ✅ LangSmith Tracing   : ON — Project: K21")
print(f"   ✅ AWS Bedrock         : Connected (us-east-1)")
print(f"   ✅ Language Model      : amazon.nova-lite-v1:0")
print(f"   ✅ Embeddings          : amazon.titan-embed-text-v2:0")
print(f"   ✅ Database            : SQLite Chinook — {len(db.get_usable_table_names())} tables")
print(f"   ✅ SQL Chain           : SQLDatabaseChain operational")

print("\n📊 DATABASE TABLES:")
for t in db.get_usable_table_names():
    print(f"   • {t}")

print("\n🎯 CAPABILITIES:")
print("   • Natural language → SQL → Answer")
print("   • Multi-table JOINs and aggregations")
print("   • Business intelligence queries")
print("   • Full LangSmith trace logging")

print("\n🔍 FINAL VERIFICATION TEST:")
final_q = "Show me the top 3 artists by number of albums"
print(f"Query: '{final_q}'")
try:
    result = sql_chain.invoke({"query": final_q})
    print(f"✅ Answer: {result.get('result', 'No result')}")
    print("\n🎉 SQL RAG SYSTEM WORKING PERFECTLY!")
except Exception as e:
    print(f"❌ Error: {e}")

print(f"\n🔗 Traces: https://smith.langchain.com → Projects → K21")
print("=" * 60)

📋 SQL RAG SYSTEM — FINAL SUMMARY

🔧 COMPONENT STATUS:
   ✅ LangSmith Tracing   : ON — Project: K21
   ✅ AWS Bedrock         : Connected (us-east-1)
   ✅ Language Model      : amazon.nova-lite-v1:0
   ✅ Embeddings          : amazon.titan-embed-text-v2:0
   ✅ Database            : SQLite Chinook — 11 tables
   ✅ SQL Chain           : SQLDatabaseChain operational

📊 DATABASE TABLES:
   • Album
   • Artist
   • Customer
   • Employee
   • Genre
   • Invoice
   • InvoiceLine
   • MediaType
   • Playlist
   • PlaylistTrack
   • Track

🎯 CAPABILITIES:
   • Natural language → SQL → Answer
   • Multi-table JOINs and aggregations
   • Business intelligence queries
   • Full LangSmith trace logging

🔍 FINAL VERIFICATION TEST:
Query: 'Show me the top 3 artists by number of albums'


> Entering new SQLDatabaseChain chain...
Show me the top 3 artists by number of albums
SQLQuery:SELECT a.Name, COUNT(DISTINCT al.AlbumId) AS NumberOfAlbums
FROM Artist a
JOIN Album al ON a.ArtistId = al.ArtistId
GROUP 